In [ ]:
import sys
from pathlib import Path

# Get project root - the project root should have both 'vault' and 'ensemble' directories
# Walk up from current directory until we find a directory with both
current_dir = Path.cwd()
project_root = current_dir

# Walk up the directory tree to find project root
while project_root != project_root.parent:
    # Check if this directory has both vault and ensemble (indicating project root)
    has_vault = (project_root / 'vault').exists()
    has_ensemble = (project_root / 'ensemble').exists()
    
    if has_vault and has_ensemble:
        # Found the project root
        break
    
    project_root = project_root.parent
else:
    # If we reached the filesystem root without finding it, use parent of current_dir
    # (assuming we're in research/ subdirectory)
    if (current_dir.parent / 'vault').exists() and (current_dir.parent / 'ensemble').exists():
        project_root = current_dir.parent
    else:
        # Last resort: use current directory
        project_root = current_dir

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Has vault: {(project_root / 'vault').exists()}")
print(f"Has ensemble: {(project_root / 'ensemble').exists()}")
print(f"Python path includes: {str(project_root) in sys.path}")

In [ ]:
"""
Portfolio Testing with PortfolioTester

This notebook demonstrates how to use the PortfolioTester framework to:
1. Load data and create portfolios
2. Fit portfolios using candles
3. Generate predictions at different granularity levels
4. Calculate returns and generate tearsheets
5. Analyze performance at portfolio, ensemble, and base model levels
"""

from datetime import datetime
import pandas as pd
import numpy as np

from utils.enums import TimeFrame, Ticker
from utils import helpers
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.portfolio_tester import PortfolioTester

# Portfolio Testing Framework Demo

## Overview

The `PortfolioTester` class provides a comprehensive framework for testing portfolios:
- **Automatic returns calculation**: Calculates log returns from candles automatically
- **Granular predictions**: Get predictions at portfolio, ensemble, and base model levels
- **Integrated tearsheets**: Generate QuantStats tearsheets at all granularity levels
- **Baseline comparison**: Compare against equal-weight or buy-and-hold baselines

In [ ]:
# Load data
print("Loading data...")
train_candles = helpers.load_data_multi_ticker(
    tickers=[Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY],
    timeframe=TimeFrame.D,
    start=datetime(2010, 1, 1),
    end=datetime(2020, 1, 1),
    use_millisecond_offset=True
)

test_candles = helpers.load_data_multi_ticker(
    tickers=[Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY],
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 1, 1),
    use_millisecond_offset=True
)

print(f"Training data: {len(train_candles)} candles")
print(f"Test data: {len(test_candles)} candles")
print(f"Tickers: {sorted(train_candles['ticker'].unique())}")

## Step 1: Create Portfolio

For this demo, we'll use a simple buy-and-hold ensemble. In practice, you would load your ensemble from the vault or create it from feature configs.

In [ ]:
# Load ensemble from vault using vault_manager
from ensemble.vault_manager import load_ensemble_from_vault

# Load ensemble from vault
ensemble_dir = 'vault/D/buy_hold_long'
print(f"Loading ensemble from {ensemble_dir}...")
ensemble = load_ensemble_from_vault(ensemble_dir, refit=False, target_volatility=0.10)
rsi_ensemble = load_ensemble_from_vault('vault/D/rsi_bias_lookback_2_long', refit=False, target_volatility=0.10)
print(f"Ensemble loaded: {ensemble}")
print(f"Ensemble fitted: {ensemble.is_fitted_}")
print(f"Base models: {list(ensemble.base_models.keys())}")

# Create portfolio
portfolio = Portfolio(
    ensembles=[ensemble, rsi_ensemble],
    trading_timeframe=TimeFrame.D,
    target_volatility=0.10,
    max_position_pct=2.5
)
print(f"\nPortfolio created with {len(portfolio.ensembles)} ensemble(s)")

## Step 2: Initialize PortfolioTester

The PortfolioTester wraps your portfolio and provides a convenient API for testing.

In [ ]:
# Initialize PortfolioTester
tester = PortfolioTester(
    portfolio=portfolio,
    baseline_mode='equal_weight'  # or 'buy_hold'
)

# Fit portfolio (automatically calculates log returns)
print("\nFitting portfolio...")
tester.fit(train_candles)

# Print comprehensive diagnostics
tester.portfolio.print_diagnostics()

# Also print individual values for easy access
diagnostics = tester.portfolio.get_diagnostics()
print(f"\n📋 Quick Summary:")
print(f"  IDM: {diagnostics.get('idm', 'Not calculated')}")
print(f"  FDM: {diagnostics.get('fdm', 'Not calculated')}")
print(f"  Mean Return Correlation: {diagnostics.get('mean_return_correlation', 'N/A')}")
print(f"  Mean Forecast Correlation: {diagnostics.get('mean_forecast_correlation', 'N/A')}")
print(f"  Ensemble fitted: {ensemble.is_fitted_}")

## Step 3: Generate Predictions

You can get predictions at different granularity levels:
- **Portfolio level**: Aggregated positions (default)
- **Ensemble level**: Individual ensemble predictions
- **Base model level**: Individual base model predictions

In [ ]:
# Basic prediction (portfolio level only)
print("\nGenerating portfolio-level predictions...")
positions = tester.predict(test_candles)
print(f"Generated {len(positions)} position predictions")
print(f"Position fraction range: [{positions['position_fraction'].min():.4f}, {positions['position_fraction'].max():.4f}]")

# With ensemble-level and base model-level granularity
print("\nGenerating granular predictions (ensemble + base model level)...")
results = tester.predict(
    test_candles,
    return_ensemble_predictions=True,
    return_base_model_predictions=True
)

portfolio_positions = results['portfolio']
ensemble_predictions = results['ensembles']
base_model_predictions = results['base_models']

print(f"\nGranular predictions:")
print(f"  Portfolio positions: {len(portfolio_positions)}")
print(f"  Ensemble predictions: {len(ensemble_predictions)}")
print(f"  Base model predictions: {len(base_model_predictions)}")

In [ ]:
portfolio_positions

## Step 4: Calculate Returns

Calculate strategy returns and baseline returns for comparison.

In [ ]:
# Calculate strategy returns
print("\nCalculating strategy returns...")
strategy_returns = tester.calculate_strategy_returns(test_candles)
print(f"Strategy returns: {len(strategy_returns)} days")
print(f"Total return: {strategy_returns.sum():.4f}")
if strategy_returns.std() > 0:
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {sharpe:.4f}")

# Calculate baseline returns
print("\nCalculating baseline returns...")
baseline_returns = tester.calculate_baseline_returns(test_candles)
print(f"Baseline returns: {len(baseline_returns)} days")
print(f"Total return: {baseline_returns.sum():.4f}")
if baseline_returns.std() > 0:
    baseline_sharpe = baseline_returns.mean() / baseline_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {baseline_sharpe:.4f}")

# Compare performance
print(f"\nPerformance Comparison:")
print(f"  Strategy total return: {strategy_returns.sum():.4f} ({strategy_returns.sum()*100:.2f}%)")
print(f"  Baseline total return: {baseline_returns.sum():.4f} ({baseline_returns.sum()*100:.2f}%)")
print(f"  Excess return: {(strategy_returns.sum() - baseline_returns.sum()):.4f} ({(strategy_returns.sum() - baseline_returns.sum())*100:.2f}%)")

## Step 5: Generate Tearsheets

Generate QuantStats tearsheets for performance analysis.

In [ ]:
# Generate portfolio-level tearsheet
print("\nGenerating portfolio tearsheet...")
tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio',
    output_dir='reports/portfolio',  # Save HTML report to this directory
    mode='html',  # 'html', 'full', 'basic', or 'metrics'
    candles_df=test_candles
)

# Generate ensemble-level tearsheets (if available)
if tester.ensemble_predictions:
    print("\nGenerating ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

# Generate base model-level tearsheets (if available)
if tester.base_model_predictions:
    print("\nGenerating base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

## Complete Example Workflow

Here's a complete example showing the full workflow:

In [ ]:
# Summary: Access all results
print("\n" + "="*70)
print("SUMMARY: All Results Available")
print("="*70)
print(f"Portfolio positions: {len(tester.positions_df) if tester.positions_df is not None else 0}")
print(f"Ensemble predictions: {len(tester.ensemble_predictions) if tester.ensemble_predictions else 0}")
print(f"Base model predictions: {len(tester.base_model_predictions) if tester.base_model_predictions else 0}")
print(f"Strategy returns: {len(tester.strategy_returns) if tester.strategy_returns is not None else 0}")
print(f"Baseline returns: {len(tester.baseline_returns) if tester.baseline_returns is not None else 0}")

# Example: Access specific results
if tester.positions_df is not None:
    print(f"\nPortfolio positions sample:")
    print(tester.positions_df.head())
    
if tester.strategy_returns is not None:
    print(f"\nStrategy returns statistics:")
    print(f"  Mean daily return: {tester.strategy_returns.mean():.6f}")
    print(f"  Std daily return: {tester.strategy_returns.std():.6f}")
    print(f"  Min return: {tester.strategy_returns.min():.6f}")
    print(f"  Max return: {tester.strategy_returns.max():.6f}")